# Repeat-seed results

Reads the job logs under `logs/` and builds the summary table averaged over its seeds.

In [213]:
import base64, glob, math, re, statistics, subprocess, tempfile
from collections import defaultdict
from pathlib import Path

from IPython.display import HTML, display
from scipy.stats import sem, t

In [214]:
CONFIGS = ["C1", "C2", "C3"]
QUALITY = ["Macro precision", "Macro recall", "Micro precision", "Micro recall"]

# Log label -> (scale, decimals shown), in column order.
COLUMNS = {"counterexamples": (1, 1),
           "Total membership queries": (1, 0),
           "Total time (ms)": (1 / 60000, 1),         # shown in minutes
           **{label: (1, 3) for label in QUALITY}}

# Log folders are logs/<model>/<sampler>_<precomp|noprecomp>/<config>/, keyed by MODEL_NAME.
MODELS = {"mistral": "mistral-7b-instruct-v0.3",
          "deepseek-r1-14b": "DeepSeek-R1-Distill-Qwen-14B",
          "deepseek-r1-32b": "DeepSeek-R1-Distill-Qwen-32B"}



def models_in(results, sampler, precomp):
    """The models with a log folder under this sampler and precomp, in MODELS order."""
    found = {model for _, model, s, p in results if (s, p) == (sampler, precomp)}
    order = list(MODELS)
    return sorted(found, key=lambda m: (order.index(m) if m in order else len(order), m))


def figures(text):
    """Each COLUMNS figure logged in text, scaled; the last one when logged twice,
    as a precomp run evaluates after precomputation and again after learning."""
    out = {}
    for label, (scale, _) in COLUMNS.items():
        hits = re.findall(rf"^{re.escape(label)}: ([\d.]+)$", text, re.M)
        out[label] = float(hits[-1]) * scale if hits else None
    return out


def parse(path):
    text = open(path, errors="replace").read()
    model, arm, config_name = Path(path).parent.parts[-3:]
    sampler, precomp = arm.split("_")
    config, prompt = config_name.split("-", 1)
    row = {
        "path": path,
        "config": config.upper(),
        "prompt": prompt,
        "precomp": precomp == "precomp",
        "model": model,
        "sampler": sampler,
        "done": "Ontology learned successfully!" in text,
        **figures(text),
    }
    # Precomputation counts one membership query per ordered class pair, and the
    # run's totals include it and its evaluation; the model rows show the loop alone.
    pairs = re.search(r"^PRECOMPUTATION.*?: (\d+) of (\d+) ordered class pairs", text, re.M)
    row["precomputation"] = None
    if pairs:
        added, queries = int(pairs[1]), int(pairs[2])
        # Logged since 2026-09-11; an older run's time cannot be split.
        pre_ms = re.search(r"^Precomputation time \(ms\): (\d+)$", text, re.M)
        eval_ms = re.search(r"^Precomputation evaluation time \(ms\): (\d+)$", text, re.M)
        scale = COLUMNS["Total time (ms)"][0]
        fetched = re.search(r"^Batch pre-warm: .*?(\d+) to fetch", text, re.M)
        row["precomputation"] = {"counterexamples": added,    # shown as the axioms it added
                                 "Total membership queries": queries,
                                 "Total time (ms)": int(pre_ms[1]) * scale if pre_ms else None,
                                 # Whether it asked the LLM, or only read the cache.
                                 "fetched": bool(fetched and int(fetched[1]))}
        if row["Total membership queries"] is not None:
            row["Total membership queries"] -= queries
        if pre_ms and eval_ms and row["Total time (ms)"] is not None:
            row["Total time (ms)"] -= (int(pre_ms[1]) + int(eval_ms[1])) * scale
        # Only the evaluation: the text up to the next one also holds the loop's totals.
        pre = re.search(r"after precomputation(.*?)=== BARIS EVALUATION", text, re.S)
        if pre:
            row["precomputation"].update(
                {m: v for m, v in figures(pre[1]).items() if m in QUALITY})
    # Unanchored: vLLM's progress bar ends on a carriage return, so a
    # counterexample announced after one shares its physical line.
    row["counterexamples"] = len(re.findall(r"Counterexample \d+ at sample", text))
    return row

In [215]:
def cell(values, digits):
    """Mean and 95% t interval over the runs that logged this figure."""
    values = [v for v in values if v is not None]
    if not values:
        return "--"
    mean = statistics.mean(values)
    if len(values) < 2:
        # A single run's count prints as logged: 20, not 20.0.
        return str(mean) if isinstance(mean, int) else f"{mean:.{digits}f}"
    ci = t.ppf(0.975, len(values) - 1) * sem(values)
    return rf"{mean:.{digits}f}\,{{\scriptsize $\pm${ci:.{digits}f}}}"

In [216]:
def load():
    """-> {(config, model, sampler, precomp): [finished runs]}, one run per log.
    Every folder gets a key, so one whose jobs all died shows as dashes."""
    results = defaultdict(list)
    for path in sorted(glob.glob("../logs/*/*/*/exactlearner-*.log")):
        row = parse(path)
        runs = results[row["config"], row["model"], row["sampler"], row["precomp"]]
        # A run stopped at walltime has no final hypothesis to evaluate.
        if row["done"]:
            runs.append(row)
    return dict(results)

In [217]:
HEADER = r""" &  & \multicolumn{3}{c}{\textbf{Learning cost}} & \multicolumn{4}{c}{\textbf{Learned quality}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-9}
 &  &  & \textbf{Mem.} & \textbf{Time} & \multicolumn{1}{l}{\textbf{Macro}} & \multicolumn{1}{l}{\textbf{Macro}} & \multicolumn{1}{l}{\textbf{Micro}} & \multicolumn{1}{l@{}}{\textbf{Micro}} \\
\textbf{Config} & \textbf{n} & \textbf{CEs} & \textbf{queries} & \textbf{(min)} & \multicolumn{1}{l}{\textbf{Precision}} & \multicolumn{1}{l}{\textbf{Recall}} & \multicolumn{1}{l}{\textbf{Precision}} & \multicolumn{1}{l@{}}{\textbf{Recall}} \\"""

N_COLS = 9
SAMPLERS = ["weighted", "unweighted"]


def arms_in(results, precomp, models=None):
    """(model, sampler) pairs with a log folder under precomp: models in the given
    order (default MODELS order), then samplers in SAMPLERS order."""
    found = {(m, s) for _, m, s, p in results if p == precomp and (not models or m in models)}
    rank = lambda order, x: (order.index(x) if x in order else len(order), x)
    return sorted(found, key=lambda ms: (rank(models or list(MODELS), ms[0]),
                                         rank(SAMPLERS, ms[1])))


def runs_per_config(results, model, sampler, precomp):
    return [results.get((config, model, sampler, precomp)) for config in CONFIGS]


def per_config(row, runs_per_config):
    """row(runs) gives the cells after Config; None where no run finished."""
    return [row(runs) if runs else None for runs in runs_per_config]


def row_cells(runs):
    """n, then one cell per column."""
    return [str(len(runs))] + [cell([r[label] for r in runs], digits)
                               for label, (_, digits) in COLUMNS.items()]


SHARED = ["counterexamples", "Total membership queries", "Total time (ms)"]


def precomp_block(runs_per_config):
    """Precomputation rows, one per config. It asks only about class names, so
    configs with one signature share a single precomputation (n=1): when they
    agree on it, its axioms added, queries and time are set once across the
    rows. The scores stay per config, as each is measured on its own baseSet."""
    pres = [[r["precomputation"] for r in runs or [] if r["precomputation"]]
            for runs in runs_per_config]
    if not any(pres):
        return [None] * len(pres)

    def figure(ps, label, digits):
        # Every run replays the same precomputation: identical values count once.
        values = [p[label] for p in ps if not (label == "Total time (ms)" and not p["fetched"])]
        return cell(values if len(set(values)) > 1 else values[:1], digits)

    digits = {label: d for label, (_, d) in COLUMNS.items()}
    everyone = [p for ps in pres for p in ps]
    merged = all(pres) and all(len({p[label] for p in everyone}) == 1 for label in SHARED[:2])
    rows = []
    for i, ps in enumerate(pres):
        if not ps:
            rows.append(None)
            continue
        # The time is the run that asked the LLM; the rest only read its answers back.
        front = ["1"] + [figure(everyone if merged else ps, label, digits[label]) for label in SHARED]
        if merged:
            front = [rf"\multirow{{{len(pres)}}}{{*}}{{{c}}}" if i == 0 else "" for c in front]
        # Left-aligned so each score sits under the model rows' means, not their
        # intervals; the last column keeps the tabular's closing @{}.
        specs = ["l"] * (len(QUALITY) - 1) + ["l@{}"]
        rows.append(front + [rf"\multicolumn{{1}}{{{s}}}{{{figure(ps, label, digits[label])}}}"
                             for s, label in zip(specs, QUALITY)])
    return rows


def table_latex(blocks, header, caption, label):
    """blocks: (title, rows), rows holding per config the cells after Config, or
    None for dashes."""
    out = [r"\begin{table*}[]", r"\centering", r"\setlength{\tabcolsep}{4pt}",
           r"\begin{tabular}{@{}lrrrrrrrr@{}}", r"\toprule", header, r"\midrule"]
    for i, (title, rows) in enumerate(blocks):
        if i:
            out.append(r"\addlinespace")
        out.append(rf"\multicolumn{{{N_COLS}}}{{@{{}}l}}{{\itshape {title}}} \\")
        for config, cells in zip(CONFIGS, rows):
            out.append(" & ".join([config] + (cells or ["--"] * (N_COLS - 1))) + r" \\")
    out += [r"\bottomrule", r"\end{tabular}", rf"\caption{{{caption}}}",
            rf"\label{{{label}}}", r"\end{table*}"]
    return "\n".join(out)


def summary_latex(results, precomp=False, models=None):
    """One block per model-sampler pair. models: folder keys to show, in order;
    default every model with a folder."""
    arms = arms_in(results, precomp, models)
    blocks = []
    for model in dict.fromkeys(m for m, _ in arms):
        name = MODELS.get(model, model)
        samplers = [s for m, s in arms if m == model]
        if precomp:
            # The sampler plays no part in precomputation: one block per model, over every arm's runs.
            pooled = [sum((results.get((c, model, s, True)) or [] for s in samplers), [])
                      for c in CONFIGS]
            blocks.append((f"Precomputation: {name}", precomp_block(pooled)))
        blocks += [(f"{name}-{s}", per_config(row_cells, runs_per_config(results, model, s, precomp)))
                   for s in samplers]
    # Caption: "nlp-advanced, precomp=False", prompt read off the runs shown.
    shown = [r for m, s in arms for runs in runs_per_config(results, m, s, precomp) for r in runs or []]
    prompt = ", ".join(sorted({r["prompt"] for r in shown}))
    caption = f"{prompt}, precomp={precomp}"
    if precomp:
        # Time is split only in runs logged since 2026-09-11.
        timed = all(r["precomputation"] and r["precomputation"]["Total time (ms)"] is not None
                    for r in shown)
        caption += (". In the precomputation rows CEs is the axioms it added; the model"
                    f" rows' queries{' and time' if timed else ''} are the learning loop's alone")
    return table_latex(blocks, HEADER, caption, "table:repeats" + ("-precomp" if precomp else ""))

In [218]:
PREAMBLE = r"""\documentclass[border=6pt,varwidth=40cm]{standalone}
\usepackage{booktabs,caption,multirow,threeparttable}
% standalone cannot hold a float; threeparttable sets the caption as wide as the table.
\renewenvironment{table*}[1][]{\begin{threeparttable}}{\end{threeparttable}}
\begin{document}
"""

ZOOM_FACTOR = 1.5  # one factor for every table, so type matches across them


def show(src):
    """Render the table as SVG; print the LaTeX if it does not compile."""
    with tempfile.TemporaryDirectory() as tmp:
        (Path(tmp) / "t.tex").write_text(PREAMBLE + src + "\n\\end{document}\n")
        ok = lambda *cmd: subprocess.run(cmd, cwd=tmp, capture_output=True).returncode == 0
        # --no-fonts draws glyphs as paths: browsers do not render SVG fonts.
        if (ok("latex", "-interaction=nonstopmode", "-halt-on-error", "t.tex")
                and ok("dvisvgm", "--exact-bbox", "--bbox=4pt", "--no-fonts",
                       f"--zoom={ZOOM_FACTOR}", "t.dvi", "-o", "t.svg")):
            svg = base64.b64encode((Path(tmp) / "t.svg").read_bytes()).decode()
            # An <img> keeps each SVG's glyph ids apart; white keeps dark themes legible.
            display(HTML(f'<img src="data:image/svg+xml;base64,{svg}" style="background:#fff">'))
            return
    print(src)

## Results

In [219]:
results = load()

In [220]:
repeats_tex = summary_latex(results, precomp=False, models=["mistral"])
show(repeats_tex)
print(repeats_tex)

\begin{table*}[]
\centering
\setlength{\tabcolsep}{4pt}
\begin{tabular}{@{}lrrrrrrrr@{}}
\toprule
 &  & \multicolumn{3}{c}{\textbf{Learning cost}} & \multicolumn{4}{c}{\textbf{Learned quality}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-9}
 &  &  & \textbf{Mem.} & \textbf{Time} & \multicolumn{1}{l}{\textbf{Macro}} & \multicolumn{1}{l}{\textbf{Macro}} & \multicolumn{1}{l}{\textbf{Micro}} & \multicolumn{1}{l@{}}{\textbf{Micro}} \\
\textbf{Config} & \textbf{n} & \textbf{CEs} & \textbf{queries} & \textbf{(min)} & \multicolumn{1}{l}{\textbf{Precision}} & \multicolumn{1}{l}{\textbf{Recall}} & \multicolumn{1}{l}{\textbf{Precision}} & \multicolumn{1}{l@{}}{\textbf{Recall}} \\
\midrule
\multicolumn{9}{@{}l}{\itshape mistral-7b-instruct-v0.3-weighted} \\
C1 & 10 & 9.9\,{\scriptsize $\pm$1.8} & 481\,{\scriptsize $\pm$124} & 0.4\,{\scriptsize $\pm$0.0} & 0.977\,{\scriptsize $\pm$0.009} & 0.753\,{\scriptsize $\pm$0.003} & 0.968\,{\scriptsize $\pm$0.004} & 0.482\,{\scriptsize $\pm$0.005} \\
C2 & 10 & 45.

In [221]:
repeats_precomp_tex = summary_latex(results, precomp=True, models=["mistral"])
show(repeats_precomp_tex)
print(repeats_precomp_tex)

\begin{table*}[]
\centering
\setlength{\tabcolsep}{4pt}
\begin{tabular}{@{}lrrrrrrrr@{}}
\toprule
 &  & \multicolumn{3}{c}{\textbf{Learning cost}} & \multicolumn{4}{c}{\textbf{Learned quality}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-9}
 &  &  & \textbf{Mem.} & \textbf{Time} & \multicolumn{1}{l}{\textbf{Macro}} & \multicolumn{1}{l}{\textbf{Macro}} & \multicolumn{1}{l}{\textbf{Micro}} & \multicolumn{1}{l@{}}{\textbf{Micro}} \\
\textbf{Config} & \textbf{n} & \textbf{CEs} & \textbf{queries} & \textbf{(min)} & \multicolumn{1}{l}{\textbf{Precision}} & \multicolumn{1}{l}{\textbf{Recall}} & \multicolumn{1}{l}{\textbf{Precision}} & \multicolumn{1}{l@{}}{\textbf{Recall}} \\
\midrule
\multicolumn{9}{@{}l}{\itshape Precomputation: mistral-7b-instruct-v0.3} \\
C1 & \multirow{3}{*}{1} & \multirow{3}{*}{452} & \multirow{3}{*}{17030} & \multirow{3}{*}{--} & \multicolumn{1}{l}{0.732} & \multicolumn{1}{l}{0.828} & \multicolumn{1}{l}{0.832} & \multicolumn{1}{l@{}}{0.494} \\
C2 &  &  &  &  & \multicolumn{1

## Counterexample arrival rate

Whether counterexamples arrive more slowly later in a run. The gap is the samples
drawn since the previous one; each seed is one replicate, so the trend test is a
per-seed rank correlation with the counterexample index and a sign test over the
seeds. A run submitted on its own is one replicate, at n=1. Weighted arm only; pass
`sampler="unweighted"` to `arrival_latex` for the other one.

In [222]:
CE_LINE = re.compile(r"Counterexample \d+ at sample (\d+) \(\+(\d+) since the last one\)")
BUDGET = re.compile(r"PAC sample budget \(numberOfSamples\) = (\d+)")

QUARTERS = 4


def arrivals(path):
    """-> (gaps between counterexamples, sample of the last one, budget)."""
    text = open(path, errors="replace").read()
    ces = [(int(s), int(g)) for s, g in CE_LINE.findall(text)]
    budget = BUDGET.search(text)
    return ([g for _, g in ces], ces[-1][0] if ces else 0,
            int(budget[1]) if budget else None)





In [223]:
def arrival_cells(runs):
    """n, median CEs, mean gap per quarter pooled over seeds, Q4/Q1 and tail."""
    runs = [arrivals(r["path"]) for r in runs]
    runs = [(g, last, budget) for g, last, budget in runs if len(g) >= QUARTERS]
    if not runs:
        return None

    quarters = []
    for q in range(QUARTERS):
        pooled = [g[len(g) * q // QUARTERS:len(g) * (q + 1) // QUARTERS] for g, _, _ in runs]
        quarters.append(statistics.mean([x for chunk in pooled for x in chunk]))

    # Budget left unspent after the last counterexample: high means the run ran
    # out of things to learn long before it ran out of budget.
    tail = statistics.median([(b - last) / b for _, last, b in runs if b])
    ces = statistics.median([len(g) for g, _, _ in runs])
    return ([str(len(runs)), f"{ces:.0f}"] + [f"{q:.0f}" for q in quarters]
            + [f"{quarters[-1] / quarters[0]:.2f}", f"{tail * 100:.0f}\\%"])


ARRIVAL_HEADER = r""" &  &  & \multicolumn{4}{c}{\textbf{Mean gap}} & \multicolumn{2}{c}{\textbf{Trend}} \\
\cmidrule(lr){4-7} \cmidrule(lr){8-9}
\textbf{Config} & \textbf{n} & \textbf{CEs} & \textbf{Q1} & \textbf{Q2} & \textbf{Q3} & \textbf{Q4} & \textbf{Q4/Q1} & \textbf{tail} \\"""


def arrival_latex(results, sampler="weighted", precomp=False, models=None):
    """models: folder keys to show, in order; default every model with a folder."""
    blocks = [(MODELS.get(m, m), per_config(arrival_cells, runs_per_config(results, m, sampler, precomp)))
              for m in models or models_in(results, sampler, precomp)]
    return table_latex(blocks, ARRIVAL_HEADER,
                       "Mean samples between counterexamples, by quarter of each run, "
                       "pooled over the replicates. Tail is the unspent budget. "
                       "A gap cannot exceed the budget left, biasing late quarters down.",
                       "table:arrivals")


In [224]:
arrival_tex = arrival_latex(results, models=["mistral"])
show(arrival_tex)